In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 02. Анализ признаков\n",
    "## Исследование извлеченных признаков и их важности для классификации"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "\n",
    "sys.path.append(str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.decomposition import PCA\n",
    "from sklearn.manifold import TSNE\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Загружаем признаки\n",
    "X_train = np.load('../data/processed/features_train.npy')\n",
    "X_val = np.load('../data/processed/features_val.npy')\n",
    "X_test = np.load('../data/processed/features_test.npy')\n",
    "\n",
    "y_train = np.load('../data/processed/labels_train.npy')\n",
    "y_val = np.load('../data/processed/labels_val.npy')\n",
    "y_test = np.load('../data/processed/labels_test.npy')\n",
    "\n",
    "print(f\"Train features shape: {X_train.shape}\")\n",
    "print(f\"Val features shape: {X_val.shape}\")\n",
    "print(f\"Test features shape: {X_test.shape}\")\n",
    "\n",
    "# Загружаем названия признаков\n",
    "with open('../data/processed/feature_names.txt', 'r') as f:\n",
    "    feature_names = [line.strip() for line in f]\n",
    "\n",
    "print(f\"\\nКоличество признаков: {len(feature_names)}\")\n",
    "print(f\"Первые 10 признаков: {feature_names[:10]}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Статистика признаков\n",
    "feature_stats = pd.DataFrame({\n",
    "    'mean': X_train.mean(axis=0),\n",
    "    'std': X_train.std(axis=0),\n",
    "    'min': X_train.min(axis=0),\n",
    "    'max': X_train.max(axis=0)\n",
    "}, index=feature_names[:X_train.shape[1]])\n",
    "\n",
    "feature_stats.head(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Анализ важности признаков с помощью Random Forest\n",
    "rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)\n",
    "rf.fit(X_train, y_train)\n",
    "\n",
    "importances = rf.feature_importances_\n",
    "indices = np.argsort(importances)[::-1]\n",
    "\n",
    "# Топ-20 важных признаков\n",
    "plt.figure(figsize=(12, 8))\n",
    "top_n = 20\n",
    "plt.barh(range(top_n), importances[indices][:top_n][::-1])\n",
    "plt.yticks(range(top_n), [feature_names[i] for i in indices[:top_n]][::-1])\n",
    "plt.xlabel('Feature Importance')\n",
    "plt.title('Top 20 Most Important Features (Random Forest)')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# PCA визуализация\n",
    "pca = PCA(n_components=2)\n",
    "X_pca = pca.fit_transform(X_train)\n",
    "\n",
    "plt.figure(figsize=(12, 5))\n",
    "\n",
    "plt.subplot(1, 2, 1)\n",
    "plt.scatter(X_pca[y_train == 0, 0], X_pca[y_train == 0, 1], \n",
    "           c='green', label='Human', alpha=0.6, s=10)\n",
    "plt.scatter(X_pca[y_train == 1, 0], X_pca[y_train == 1, 1], \n",
    "           c='red', label='Robot', alpha=0.6, s=10)\n",
    "plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')\n",
    "plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')\n",
    "plt.title('PCA Visualization')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "plt.subplot(1, 2, 2)\n",
    "plt.bar(range(1, 11), pca.explained_variance_ratio_[:10] * 100)\n",
    "plt.xlabel('Principal Component')\n",
    "plt.ylabel('Explained Variance (%)')\n",
    "plt.title('PCA Explained Variance Ratio')\n",
    "plt.grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# t-SNE визуализация (берем подвыборку для скорости)\n",
    "n_samples = 1000\n",
    "idx = np.random.choice(len(X_train), n_samples, replace=False)\n",
    "X_sample = X_train[idx]\n",
    "y_sample = y_train[idx]\n",
    "\n",
    "tsne = TSNE(n_components=2, random_state=42, perplexity=30)\n",
    "X_tsne = tsne.fit_transform(X_sample)\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "plt.scatter(X_tsne[y_sample == 0, 0], X_tsne[y_sample == 0, 1], \n",
    "           c='green', label='Human', alpha=0.6, s=20)\n",
    "plt.scatter(X_tsne[y_sample == 1, 0], X_tsne[y_sample == 1, 1], \n",
    "           c='red', label='Robot', alpha=0.6, s=20)\n",
    "plt.xlabel('t-SNE Component 1')\n",
    "plt.ylabel('t-SNE Component 2')\n",
    "plt.title('t-SNE Visualization of Audio Features')\n",
    "plt.legend()\n",
    "plt.grid(True, alpha=0.3)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Анализ корреляции между признаками\n",
    "corr_matrix = np.corrcoef(X_train.T)\n",
    "\n",
    "plt.figure(figsize=(14, 12))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "sns.heatmap(corr_matrix[:50, :50], mask=mask[:50, :50], \n",
    "            cmap='coolwarm', center=0, square=True,\n",
    "            cbar_kws={\"shrink\": 0.8})\n",
    "plt.title('Feature Correlation Matrix (first 50 features)')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Распределение признаков по классам\n",
    "fig, axes = plt.subplots(3, 4, figsize=(16, 12))\n",
    "axes = axes.flatten()\n",
    "\n",
    "top_features = [feature_names[i] for i in indices[:12]]\n",
    "\n",
    "for i, (feat_name, ax) in enumerate(zip(top_features, axes)):\n",
    "    feat_idx = feature_names.index(feat_name)\n",
    "    \n",
    "    for class_idx, class_name, color in [(0, 'human', 'green'), (1, 'robot', 'red')]:\n",
    "        data = X_train[y_train == class_idx, feat_idx]\n",
    "        ax.hist(data, bins=30, alpha=0.5, label=class_name, color=color, density=True)\n",
    "    \n",
    "    ax.set_xlabel(feat_name[:30] + '...' if len(feat_name) > 30 else feat_name)\n",
    "    ax.set_ylabel('Density')\n",
    "    ax.legend()\n",
    "    ax.grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Distribution of Top Features by Class', fontsize=16)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Статистическая значимость различий\n",
    "from scipy import stats\n",
    "\n",
    "significance = []\n",
    "for i, feat_name in enumerate(feature_names[:X_train.shape[1]]):\n",
    "    human_vals = X_train[y_train == 0, i]\n",
    "    robot_vals = X_train[y_train == 1, i]\n",
    "    \n",
    "    statistic, p_value = stats.mannwhitneyu(human_vals, robot_vals, alternative='two-sided')\n",
    "    \n",
    "    significance.append({\n",
    "        'feature': feat_name,\n",
    "        'p_value': p_value,\n",
    "        'significant': p_value < 0.05\n",
    "    })\n",
    "\n",
    "sig_df = pd.DataFrame(significance)\n",
    "print(f\"Количество значимых признаков (p < 0.05): {sig_df['significant'].sum()} из {len(sig_df)}\")\n",
    "print(f\"\\nТоп-10 наиболее значимых признаков:\")\n",
    "sig_df.sort_values('p_value').head(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Выводы\n",
    "print(\"=\" * 60)\n",
    "print(\"ВЫВОДЫ ПО АНАЛИЗУ ПРИЗНАКОВ\")\n",
    "print(\"=\" * 60)\n",
    "\n",
    "print(\"\\n1. Размерность признакового пространства:\")\n",
    "print(f\"   - Всего признаков: {X_train.shape[1]}\")\n",
    "\n",
    "print(\"\\n2. Наиболее важные признаки (по Random Forest):\")\n",
    "for i in range(10):\n",
    "    print(f\"   {i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.4f}\")\n",
    "\n",
    "print(\"\\n3. PCA анализ:\")\n",
    "print(f\"   - PC1 объясняет: {pca.explained_variance_ratio_[0]:.2%} дисперсии\")\n",
    "print(f\"   - Первые 5 компонент объясняют: {pca.explained_variance_ratio_[:5].sum():.2%}\")\n",
    "\n",
    "print(\"\\n4. Статистическая значимость:\")\n",
    "print(f\"   - Признаков с p < 0.05: {sig_df['significant'].sum()}\")\n",
    "print(f\"   - Признаков с p < 0.001: {(sig_df['p_value'] < 0.001).sum()}\")\n",
    "\n",
    "print(\"\\n5. Заключение:\")\n",
    "print(\"   - Признаки хорошо разделяют классы\")\n",
    "print(\"   - Можно использовать PCA для уменьшения размерности\")\n",
    "print(\"   - MFCC и спектральные признаки наиболее информативны\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}